# Ähnlichkeit, Clustering und Topic Models

Im Skript 03 haben wir Korpora verglichen, die wir vorab festgelegt hatten: die Politik-Ressorts von Der Standard gegen jene der Süddeutschen Zeitung. In diesem Skript lassen wir die Struktur aus den Dokumenten selbst entstehen. Sobald jedes Dokument ein Vektor ist, können wir messen, wie ähnlich sich zwei Dokumente sind, ähnliche Dokumente gruppieren, ohne dass uns jemand die Gruppen vorgibt, und ein Korpus in Themen zerlegen. Das sind **unüberwachte** Methoden: Es fließen keine Labels ein.

Am Ende dieses Skripts kannst du:

- die Ähnlichkeit von Dokumenten mit der **Kosinus-Ähnlichkeit** über TF-IDF-Vektoren messen;
- **dieselbe Geschichte** in zwei Zeitungen finden und einen Schwellenwert durch Lesen kalibrieren;
- die Dimensionalität mit **Latent Semantic Analysis (LSA)** reduzieren;
- Dokumente mit k-means **clustern** und die Cluster beschreiben;
- **Topic Models** (NMF und LDA) schätzen und interpretieren und beurteilen, wie weit man ihnen trauen kann;
- unüberwachte Ergebnisse mit vorhandenen **Labels** - den Ressorts der Redaktionen - gegenprüfen, ohne sie zum Lernen zu verwenden.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Visualisierung und Datentransformation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Paket für die linguistische Annotation
import spacy

# Pakete für Vektorisierung, Ähnlichkeit, Cluster Analyse und Topic Models
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, LatentDirichletAllocation, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import normalized_mutual_info_score, silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Daten

Wir laden das Korpus und das spaCy-Modell und lemmatisieren die Schlagzeilen wie in den vorherigen Skripten. Behalten werden nur Inhaltswörter: keine Satzzeichen, Zahlen und Stoppwörter. Das Parsen wird zwischengespeichert, sodass ein erneuter Durchlauf schnell geht.

In [ ]:
# Lade das Korpus und das spaCy-Modell
corpus = tm.load_corpus()
nlp = spacy.load("de_core_news_md")


# Behalte nur Inhaltswörter: keine Satzzeichen, Zahlen, Symbole oder Stoppwörter
def is_content(token) -> bool:
    return token.pos_ not in {"PUNCT", "NUM", "SPACE", "SYM", "X"} and not token.is_stop and any(
        ch.isalpha() for ch in token.text
    )


# Gib die kleingeschriebenen Lemmata der Inhaltswörter eines Dokuments zurück
def lemmas(doc) -> list[str]:
    return [t.lemma_.casefold() for t in doc if is_content(t)]


# Parse die Schlagzeilen (mit Cache) und speichere die Lemmata als neue Spalte
title_docs = tm.parse_corpus(nlp, corpus["title"].tolist(), name="titles")
corpus["title_lemmas"] = [lemmas(d) for d in title_docs]


# Die Dokumente sind bereits tokenisiert: der Vectorizer soll die Token-Listen unverändert übernehmen
def identity(tokens):
    return tokens

## Dokumente als Vektoren, Ähnlichkeit als Winkel

Im Skript 03 haben wir jede Schlagzeile in eine Zeile einer TF-IDF-Matrix verwandelt. Jede Zeile ist ein **Vektor** in einem Raum mit einer Dimension pro Term. Zwei Dokumente, die dieselben charakteristischen Wörter verwenden, zeigen in ähnliche Richtungen.

Die **Kosinus-Ähnlichkeit** misst den Winkel zwischen zwei Vektoren: 1 bei identischer Richtung, 0, wenn es keine gemeinsamen Terme gibt. Sie ignoriert die Länge der Vektoren, sodass auch ein langer und ein kurzer Text zum selben Thema hoch abschneiden können. TF-IDF-Vektoren sind bereits auf die Länge 1 normalisiert, und für solche Einheitsvektoren ist der Kosinus einfach das Skalarprodukt.

In [ ]:
# Erstelle die TF-IDF-Matrix der Schlagzeilen
tfidf = TfidfVectorizer(analyzer=identity, min_df=2)
X_titles = tfidf.fit_transform(corpus["title_lemmas"])


# Finde die n Schlagzeilen, die dem Dokument i am ähnlichsten sind
def most_similar(i: int, n: int = 6) -> pd.DataFrame:
    sims = cosine_similarity(X_titles[i], X_titles).ravel()
    order = sims.argsort()[::-1][1 : n + 1]  # überspringe das Dokument selbst
    return pd.DataFrame({"Ähnlichkeit": sims[order].round(2), "Zeitung": corpus["outlet"].iloc[order],
                         "Schlagzeile": corpus["title"].iloc[order]})


# Wähle die erste Schlagzeile zum Thema Obergrenze (für Asylanträge) als Abfrage
query = corpus.index[corpus["title"].str.contains("Obergrenze")][0]
print("Abfrage:", corpus.loc[query, "outlet"], "|", corpus.loc[query, "title"])
most_similar(query)

Die ersten Nachbarn passen zum Thema - der oberste ist dieselbe Meldung in der SZ, fast wortgleich. Weiter unten teilen die Treffer nur noch ein einzelnes Wort mit der Abfrage (*Juden* in einem Text über Antijudaismus, *Zentralrat* beim Zentralrat der Muslime) und landen trotzdem weit oben, weil die Abfrage-Schlagzeile kurz ist. Bag-of-Words-Ähnlichkeit sieht gemeinsames Vokabular, genau das und nur das. Zwei Schlagzeilen über dasselbe Ereignis in unterschiedlichen Worten - *"Kanzlerin"* in der einen, *"Merkel"* in der anderen - haben nichts gemeinsam und erhalten den Wert null. Diesen blinden Fleck sollten wir im Hinterkopf behalten; Skript 06 widmet sich ihm.

## Dieselbe Geschichte in zwei Redaktionen

Eine konkrete Aufgabe: Gibt es zu einer Schlagzeile von Der Standard eine Schlagzeile der Süddeutschen Zeitung über dieselbe Geschichte? Wenn wir die Kandidaten auf ein Zeitfenster beschränken, bleibt der Vergleich sinnvoll und der Rechenaufwand gering. Da die SZ-Daten nur den Monat kennen, ist das Fenster der **Kalendermonat**: Jede Standard-Schlagzeile wird mit allen SZ-Schlagzeilen desselben Monats verglichen. Das ist großzügiger als nötig - eine Meldung und ihr Gegenstück erscheinen meist am selben oder am nächsten Tag - und macht die Aufgabe schwerer, weil mehr falsche Kandidaten im Rennen sind.

In [ ]:
# Positionen der Standard- und der SZ-Schlagzeilen
std = np.flatnonzero(corpus["outlet"] == "Der Standard")
sz = np.flatnonzero(corpus["outlet"] == "Süddeutsche Zeitung")
month = corpus["month"].to_numpy()

# Für jeden Monat: Ähnlichkeitsmatrix (Zeilen: Standard, Spalten: SZ) und bester SZ-Treffer je Zeile
rows = []
for m in np.unique(month):
    a, b = std[month[std] == m], sz[month[sz] == m]
    S = cosine_similarity(X_titles[a], X_titles[b])
    best = S.argmax(axis=1)
    rows.append(pd.DataFrame({"std": a, "sz": b[best], "similarity": S[np.arange(len(a)), best]}))
pairs = pd.concat(rows, ignore_index=True)
pairs["standard"] = corpus["title"].to_numpy()[pairs["std"]]
pairs["sueddeutsche"] = corpus["title"].to_numpy()[pairs["sz"]]

# Zeige die Verteilung der besten Ähnlichkeitswerte
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(pairs["similarity"], bins=40, color=tm.OUTLET_COLOURS["Der Standard"])
ax.set(xlabel="Kosinus-Ähnlichkeit des besten SZ-Treffers im selben Monat", ylabel="Standard-Schlagzeilen",
       title="Die meisten Standard-Schlagzeilen haben kein nahes Gegenstück in der SZ")
plt.show()

Wo soll der Schwellenwert für "dieselbe Geschichte" liegen? Dafür gibt es keine Formel. Der ehrliche Weg ist, **Paare auf verschiedenen Ähnlichkeitsstufen zu lesen** und zu sehen, ab wann die Treffer nicht mehr stimmen:

In [ ]:
# Zeige für jedes Ähnlichkeitsband bis zu vier zufällige Paare
for lo, hi in [(0.2, 0.3), (0.3, 0.4), (0.4, 0.5), (0.5, 0.7), (0.7, 1.01)]:
    band = pairs[(pairs["similarity"] >= lo) & (pairs["similarity"] < hi)]
    print(f"\n--- Ähnlichkeit {lo:.1f}-{hi:.1f}: {len(band)} Paare")
    for _, r in band.sample(min(4, len(band)), random_state=0).iterrows():
        print(f"  STD: {r['standard']}\n  SZ:  {r['sueddeutsche']}\n")

Das Lesen der Bänder ergibt einen brauchbaren Schwellenwert, und er liegt hoch. In den unteren Bändern teilen die Paare bloß ein Wort (*Schule*, *Prädikat*, *Vorbild*) oder eine Wendung (*… auf der Spur*). Ab etwa 0,5 erscheinen echte Treffer (*Clausnitz*, die eingestellten Ermittlungen zu Merkels Handy), aber noch neben vielen falschen; erst im obersten Band beschreibt die Mehrheit der Paare dieselbe Geschichte. Dass die Schwelle so hoch liegen muss, hat zwei Gründe: Schlagzeilen sind kurz, sodass schon ein gemeinsames seltenes Wort eine hohe Ähnlichkeit ergibt, und das Monatsfenster bietet jeder Schlagzeile tausende Kandidaten, unter denen sich fast immer einer mit einem gemeinsamen Wort findet. Wähle den Schwellenwert aufgrund dessen, was du gelesen hast, nicht aufgrund des Histogramms, und halte fest, warum.

### Eine zweite Meinung: die Ressorts

Das Lesen bleibt der Maßstab, aber wir haben noch eine unabhängige Kontrolle. Beide Redaktionen haben jeden Artikel einem Ressort zugeordnet, und Skript 01 hat die Ressorts mit Gegenstück zu gemeinsamen **Ressortgruppen** zusammengefasst. Die Zuordnung hat diese Labels nie gesehen. Wenn zwei Schlagzeilen wirklich dieselbe Geschichte erzählen, sollten sie meist in derselben Ressortgruppe stehen - eine Wahl in der Politik, ein Transfer im Sport. Bei zufällig gepaarten Schlagzeilen wäre das viel seltener.

In [ ]:
# Ressortgruppen beider Seiten eines Paars (nur Paare, bei denen beide Seiten eine Gruppe haben)
group = corpus["section_group"].to_numpy()
pairs["group_std"], pairs["group_sz"] = group[pairs["std"]], group[pairs["sz"]]
both = pairs.dropna(subset=["group_std", "group_sz"])

# Anteil gleicher Ressortgruppen je Ähnlichkeitsband
bands = pd.cut(both["similarity"], [0, 0.2, 0.3, 0.4, 0.5, 0.7, 1.01], right=False)
agreement = (both["group_std"] == both["group_sz"]).groupby(bands, observed=True).agg(["mean", "size"])
agreement.columns = ["gleiche Ressortgruppe", "Paare"]

# Zum Vergleich: Übereinstimmung bei zufälliger Paarung (Produkt der Gruppenanteile beider Zeitungen)
p_std = both["group_std"].value_counts(normalize=True)
p_sz = corpus.loc[corpus["outlet"] == "Süddeutsche Zeitung", "section_group"].value_counts(normalize=True)
print(f"Zufällige Paarung: {sum(p_std.get(g, 0) * p_sz.get(g, 0) for g in p_std.index):.0%} gleiche Ressortgruppe")
agreement.style.format({"gleiche Ressortgruppe": "{:.0%}"})

Mit wachsender Ähnlichkeit steigt der Anteil der Paare aus derselben Ressortgruppe stetig über den Zufallswert, bleibt aber selbst im obersten Band weit unter 100 %. Das bestätigt, was das Lesen der Bänder gezeigt hat, mit einer Messung, die nicht von unserem Urteil abhängt. Sie ist aber kein Ersatz für das Lesen: Dieselbe Geschichte kann in zwei Redaktionen in verschiedenen Ressorts landen (ein Konzernskandal in Wirtschaft oder Panorama), und zwei verschiedene Fußballspiele stehen beide im Sport. Die Ressorts sind eine **zweite Meinung**, keine Wahrheit.

Mit einem gewählten Schwellenwert beantworten die Treffer eine journalistische Frage: **Welche Geschichten erzählen beide Zeitungen?**

In [ ]:
# Schwellenwert für "dieselbe Geschichte", gewählt durch Lesen der Ähnlichkeitsbänder oben
THRESHOLD = 0.7
# Behalte nur Paare oberhalb des Schwellenwerts
matched = pairs[pairs["similarity"] >= THRESHOLD]
print(f"{len(matched)} Standard-Schlagzeilen mit Ähnlichkeit >= {THRESHOLD} zugeordnet "
      f"({len(matched) / len(pairs):.0%} aller Standard-Schlagzeilen)\n")

# Anteil zugeordneter Schlagzeilen je Standard-Ressort (nur Ressorts mit mindestens 200 Schlagzeilen)
section = corpus["section"].to_numpy()
share = pd.Series(pairs["similarity"] >= THRESHOLD).groupby(section[pairs["std"]]).agg(["mean", "size"])
share = share[share["size"] >= 200].sort_values("mean", ascending=False)
share.columns = ["Anteil zugeordnet", "Schlagzeilen"]
share.style.format({"Anteil zugeordnet": "{:.0%}"})

Bei diesem strengen Schwellenwert wird nur ein kleiner Teil der Standard-Schlagzeilen zugeordnet, und die Unterschiede zwischen den Ressorts sind gering. Dass die Kommentare vorne liegen, sollte misstrauisch machen: Meinungstitel sind kurz und pointiert, und ein einziges gemeinsames Schlagwort reicht für eine hohe Ähnlichkeit. Das ist eher eine Demonstration der Methode als ein Befund. Die Treffer sind eine **verzerrte Stichprobe**: Geschichten, die in ähnlichen Worten erzählt werden, lassen sich leichter zuordnen - Namen von Personen, Vereinen und Ländern helfen, Wortspiele schaden -, und das Monatsfenster, der Schwellenwert und die Wahl des *besten* Treffers pro Schlagzeile prägen das Ergebnis. Eine echte Studie würde eine Zufallsstichprobe der Treffer von Hand prüfen und die Fehlerrate neben den Anteilen berichten.

## Latent Semantic Analysis

Ein TF-IDF-Raum hat Tausende Dimensionen, die meisten davon fast leer. Die **Latent Semantic Analysis (LSA)** komprimiert ihn mit einer abgeschnittenen Singulärwertzerlegung (truncated SVD): Sie findet die wenigen Richtungen, entlang derer sich die Dokumente am stärksten unterscheiden. Terme, die häufig gemeinsam vorkommen, laden auf dieselbe Richtung, sodass LSA die Vokabularlücke teilweise überbrücken kann - zwei Dokumente können im LSA-Raum nahe beieinander liegen, ohne ein Wort zu teilen, wenn ihre Wörter anderswo gemeinsam auftreten.

Für Strukturen auf Themenebene hilft mehr Text pro Dokument, daher verwenden wir ab hier **Schlagzeile plus Teaser** (Spalte `text`).

In [ ]:
# Parse Schlagzeile plus Teaser (mit Cache) und speichere die Lemmata
text_docs = tm.parse_corpus(nlp, corpus["text"].tolist(), name="texts")
corpus["text_lemmas"] = [lemmas(d) for d in text_docs]

# Erstelle die TF-IDF-Matrix der Texte
tfidf_text = TfidfVectorizer(analyzer=identity, min_df=3, max_df=0.5)
X = tfidf_text.fit_transform(corpus["text_lemmas"])
vocab = tfidf_text.get_feature_names_out()
print(f"{X.shape[0]:,} Dokumente x {X.shape[1]:,} Terme")

# Reduziere die Matrix mit LSA auf 100 Dimensionen
svd = TruncatedSVD(n_components=100, random_state=0)
Z = normalize(svd.fit_transform(X))  # normalisiere erneut, damit Kosinus wieder = Skalarprodukt
print(f"100 Dimensionen behalten {svd.explained_variance_ratio_.sum():.0%} der Varianz")

`max_df=0.5` entfernt Terme, die in mehr als der Hälfte aller Dokumente vorkommen: eine zweite Verteidigungslinie gegen Wörter, die überall häufig sind, neben dem Stoppwort-Filter.

Jede LSA-Dimension ist eine gewichtete Mischung von Termen. Die ersten paar sind lesbar:

In [ ]:
# Zeige für die Dimensionen 1 bis 5 die Terme mit den höchsten und niedrigsten Gewichten
for k in range(1, 6):
    comp = svd.components_[k]
    pos, neg = comp.argsort()[::-1][:6], comp.argsort()[:6]
    print(f"dim {k}:  + {', '.join(vocab[pos])}\n        - {', '.join(vocab[neg])}")

Die erste Dimension (nicht gezeigt, `dim 0`) bildet vor allem die Dokumentlänge und das gesamte Termgewicht ab; ab der zweiten stellen die Dimensionen Themen einander gegenüber. Ein Plot der ersten beiden inhaltlichen Dimensionen zeigt, wie die Zeitungen den Raum besetzen:

In [ ]:
# Zeichne jedes Dokument in den LSA-Dimensionen 1 und 2, eingefärbt nach Zeitung
# (die kleinere Zeitung zuletzt, damit ihre Punkte nicht von der größeren verdeckt werden)
fig, ax = plt.subplots(figsize=(7, 5.5))
for outlet in ["Süddeutsche Zeitung", "Der Standard"]:
    idx = corpus.index[corpus["outlet"] == outlet]
    ax.scatter(Z[idx, 1], Z[idx, 2], s=3, alpha=0.3, color=tm.OUTLET_COLOURS[outlet], label=outlet)
ax.set(xlabel="LSA-Dimension 1", ylabel="LSA-Dimension 2", title="Das Korpus in zwei LSA-Dimensionen")
ax.legend(markerscale=3, frameon=False)
plt.show()

Die Grafik zeigt jedes Dokument als Punkt in den LSA-Dimensionen 1 und 2. Die Punkte bilden keine Wolke, sondern einen dichten Kern mit wenigen langen Armen: Entlang jedes Arms dominiert ein Thema immer stärker - nach rechts der FC Bayern (Dimension 1 positiv), nach links oben die Fluchtbewegung, nach unten Wirtschaftsthemen wie die Deutsche Bank (vergleiche die Terme der Dimensionen oben). Die beiden Zeitungen liegen übereinander; die rote Farbe von Der Standard ist im Kern und im Arm der Fluchtbewegung zu sehen, im Bayern-Arm fehlt sie fast ganz.

## Clustering

**k-means** teilt die Dokumente in *k* Gruppen, indem es jedes Dokument wiederholt dem nächstgelegenen Gruppenzentrum zuordnet und die Zentren in den Mittelwert ihrer Mitglieder verschiebt. k-means kennen wir bereits aus dem Skript `04-data-mining-cluster-analysis`. Das Verfahren braucht *k* im Voraus. Der **Silhouetten-Koeffizient** - wie viel näher jedes Dokument seinem eigenen Cluster ist als dem nächsten - ist eine Orientierungshilfe für die Wahl, auch wenn er bei Text selten den Ausschlag gibt.

In [ ]:
# Berechne den Silhouetten-Koeffizienten für verschiedene k (auf einer Stichprobe von 5000 Dokumenten)
scores = {}
for k in [6, 10, 14, 18, 24]:
    labels = KMeans(n_clusters=k, n_init=5, random_state=0).fit_predict(Z)
    scores[k] = silhouette_score(Z, labels, sample_size=5000, random_state=0)
pd.Series(scores, name="Silhouette").round(3).to_frame().T

Silhouetten-Koeffizienten sind bei Text typischerweise sehr niedrig, und hier steigen sie mit wachsendem *k* langsam an, ohne je ein klares Maximum zu erreichen: Dokumente bilden keine gut getrennten Kugeln, sie gehen ineinander über. Das ist eine Eigenschaft der Sprache, kein Versagen der Methode, und es bedeutet, dass *k* eine Entscheidung darüber ist, auf welcher Detailebene wir lesen wollen, und keine Tatsache, die es zu entdecken gilt.

In [ ]:
# Trainiere k-means mit 14 Clustern und speichere die Cluster-Zuordnung
K = 14
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(Z)
corpus["cluster"] = km.labels_

# Beschreibe jeden Cluster durch die Terme mit dem höchsten mittleren TF-IDF-Wert unter seinen Mitgliedern
def cluster_terms(c, n=8):
    scores = np.asarray(X[(corpus["cluster"] == c).to_numpy()].mean(axis=0)).ravel()
    return ", ".join(vocab[scores.argsort()[::-1][:n]])


# Häufigstes Ressort je Cluster (z. B. "Std: Sport" oder "SZ: sport") und sein Anteil am Cluster
label = np.where(corpus["outlet"] == "Der Standard", "Std: ", "SZ: ") + corpus["section"]
top_section = pd.crosstab(corpus["cluster"], label, normalize="index")

# Tabelle: Größe, Top-Terme, Anteil der Zeitungen und häufigstes Ressort je Cluster
summary = pd.DataFrame({
    "size": corpus["cluster"].value_counts().sort_index(),
    "Top-Terme": [cluster_terms(c) for c in range(K)],
    "häufigstes Ressort": [f"{top_section.loc[c].idxmax()} ({top_section.loc[c].max():.0%})" for c in range(K)],
})
outlet_mix = pd.crosstab(corpus["cluster"], corpus["outlet"], normalize="index").round(2)
summary.join(outlet_mix).sort_values("size", ascending=False)

Lies die Tabelle Zeile für Zeile. Manche Cluster sind saubere Themen: der FC Bayern, Sportergebnisse, Russland, Syrien und die Türkei, die Fluchtbewegung, Gerichtsprozesse, Anschläge mit Toten, Unternehmen und Technik. Manche werden von einer Zeitung dominiert, was auf ein Thema hinweist, das nur in einem Land Schlagzeilen macht - der FC-Bayern-Cluster stammt fast vollständig aus der SZ -, oder auf ein redaktionelles Format, die Lehre aus Skript 03 noch einmal. Beim Lesen der Anteile hilft der Vergleich mit dem ganzen Korpus: Die SZ stellt fast drei Viertel aller Dokumente, ein Cluster mit 70 % SZ ist also ausgewogen. Und es gibt einen großen, unscharfen Cluster mit mehr als einem Drittel aller Dokumente, der alles sammelt, was nirgendwo sonst hineinpasst. k-means muss jedes Dokument irgendwo unterbringen.

Die Spalte "häufigstes Ressort" ist wieder eine zweite Meinung: Die Cluster wurden ohne die Ressorts gebildet, und wo ein Cluster zu einem großen Teil aus einem Ressort stammt (die beiden Fußball-Cluster aus dem SZ-Sport, die Cluster zu Russland und Syrien sowie zur Fluchtbewegung aus der SZ-Politik), hat k-means eine Einteilung wiedergefunden, die auch die Redaktionen vornehmen. Eine Zahl für die gesamte Übereinstimmung liefert die **normalisierte Transinformation** (Normalized Mutual Information, NMI) zwischen Clustern und Ressortgruppen: 0 bei Unabhängigkeit, 1 bei perfekter Übereinstimmung.

In [ ]:
# NMI zwischen Clustern und Ressortgruppen (nur Dokumente mit Ressortgruppe)
has_group = corpus["section_group"].notna()
nmi = normalized_mutual_info_score(corpus.loc[has_group, "section_group"], corpus.loc[has_group, "cluster"])
# Zum Vergleich: dieselbe Rechnung mit zufällig vertauschten Clusterlabels
nmi_random = normalized_mutual_info_score(
    corpus.loc[has_group, "section_group"], corpus.loc[has_group, "cluster"].sample(frac=1, random_state=0).to_numpy()
)
print(f"NMI Cluster vs. Ressortgruppe: {nmi:.2f}  (zufällig vertauschte Cluster: {nmi_random:.2f})")

Der Wert liegt klar über dem Zufallswert, aber weit unter 1. Das ist kein Versagen: Die Cluster schneiden das Korpus nach *Themen* (eine bestimmte Krise, eine Sportart), die Ressorts nach *Zuständigkeit* in der Redaktion. Beide Einteilungen sind berechtigt, und sie überschneiden sich nur teilweise.

## Topic Models

Clustering ordnet jedes Dokument genau einer Gruppe zu. Ein **Topic Model** beschreibt dagegen jedes Dokument als *Mischung* von Topics und jedes Topic als Verteilung über Wörter. Ein Teaser über die Kosten der Flüchtlingsunterbringung und einen Koalitionsstreit kann teilweise zu einem Flucht-Topic und teilweise zu einem Innenpolitik-Topic gehören.

Zwei klassische Methoden, beide in scikit-learn:

- **NMF** (nicht-negative Matrixfaktorisierung) zerlegt die TF-IDF-Matrix in eine Dokument-Topic- und eine Topic-Term-Matrix ohne negative Gewichte - genau das macht die Teile als "Mengen" eines Topics interpretierbar.
- **LDA** (Latent Dirichlet Allocation) ist ein probabilistisches Modell dafür, wie Dokumente aus Topics erzeugt werden. Es arbeitet mit rohen Häufigkeiten, nicht mit TF-IDF.

In [ ]:
# Anzahl der Topics: eine Detailebene, die wir wählen (siehe Übung 3)
N_TOPICS = 15

# NMF auf der TF-IDF-Matrix
nmf = NMF(n_components=N_TOPICS, init="nndsvda", random_state=0, max_iter=400)
W_nmf = nmf.fit_transform(X)

# LDA auf den rohen Häufigkeiten
counts = CountVectorizer(analyzer=identity, min_df=3, max_df=0.5)
C = counts.fit_transform(corpus["text_lemmas"])
lda = LatentDirichletAllocation(n_components=N_TOPICS, learning_method="batch", random_state=0, max_iter=30)
W_lda = lda.fit_transform(C)


# Beschreibe jedes Topic durch seine n stärksten Terme
def topic_table(model, terms, n=8):
    return pd.Series(
        [", ".join(terms[comp.argsort()[::-1][:n]]) for comp in model.components_],
        name="Top-Terme",
    )


# Stelle die Topics beider Modelle nebeneinander
pd.DataFrame({"NMF": topic_table(nmf, vocab), "LDA": topic_table(lda, counts.get_feature_names_out())})

Bei kurzen Texten wie Schlagzeilen und Teasern sind die NMF-Topics meist klarer als die von LDA. Das generative Modell von LDA setzt voraus, dass jedes Dokument genug Wörter enthält, um seine Topic-Mischung zu verraten, und eine Schlagzeile mit zehn Wörtern tut das nicht. Topic Modelling für kurze Texte ist eine bekannte Schwachstelle von LDA, und NMF über TF-IDF ist für Korpora wie dieses eine vernünftige Standardwahl.

### Wer berichtet über welches Topic?

Die Dokument-Topic-Gewichte lassen sich auf natürliche Weise nach Zeitung und Ressortgruppe aggregieren. Jedes NMF-Topic wird mit seinen drei stärksten Termen beschriftet.

In [ ]:
# Beschrifte jedes Topic mit seinen drei stärksten Termen
labels = [" / ".join(vocab[c.argsort()[::-1][:3]]) for c in nmf.components_]

# Mittlere Topic-Anteile je Zeitung und Ressortgruppe ("Std" = Der Standard, "SZ" = Süddeutsche Zeitung)
column = np.where(corpus["outlet"] == "Der Standard", "Std", "SZ") + " " + corpus["section_group"].fillna("übrige")
share = pd.DataFrame(normalize(W_nmf, norm="l1"), columns=labels).groupby(column.to_numpy()).mean()

# Zeichne die Anteile als Heatmap
fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(share.T.values, cmap="Blues", aspect="auto")
ax.set_xticks(range(len(share.index)), share.index, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(labels)), labels, fontsize=8)
ax.set_title("Mittlerer Topic-Anteil je Zeitung und Ressortgruppe (NMF)")
fig.colorbar(im, ax=ax, shrink=0.7)
plt.show()

Die Heatmap zeigt für jede Kombination aus Zeitung und Ressortgruppe (Spalten) den mittleren Anteil jedes Topics (Zeilen); je dunkler, desto größer. Die Ressorts werden gut wiedergefunden: Das Sport-Topic (*gewinnen / trainer / spiel*) dominiert die Sportressorts beider Zeitungen, das Topic *frau / mann / polizei* die Panorama-Ressorts, *türkei / syrien / russland* die Politik. Das allgemeine Topic *land / neu / zeigen* ist in den Digital-Ressorts und den übrigen Ressorts groß: Es sammelt Texte ohne klares Thema. Unterschiede zwischen den Zeitungen zeigen die nationalen Themen: *österreich / prozent / wien* fast nur bei Der Standard, *fc / bayern / guardiola* fast nur im SZ-Sport.

### Wie stabil ist ein Topic Model?

Die Ausgabe eines Topic Models wirkt autoritativ, deshalb lohnt es sich zu prüfen, wie viel davon von willkürlichen Entscheidungen abhängt. Wir schätzen NMF mit einer anderen zufälligen Initialisierung neu und zählen, wie viele Topics einen nahen Partner im ersten Durchlauf finden:

In [ ]:
# Schätze NMF mit zufälliger Initialisierung neu
nmf_b = NMF(n_components=N_TOPICS, init="random", random_state=1, max_iter=1000).fit(X)
# Größte Kosinus-Ähnlichkeit jedes Topics des ersten Laufs zu einem Topic des zweiten Laufs
match = cosine_similarity(nmf.components_, nmf_b.components_).max(axis=1)
print("Ähnlichkeit jedes Topics mit seinem besten Gegenstück im zweiten Durchlauf:")
print(np.sort(match).round(2))
print(f"{(match > 0.8).sum()} von {N_TOPICS} Topics werden eng wiedergefunden (Kosinus > 0.8)")

Typischerweise taucht ein Kern von Topics fast unverändert wieder auf - das sind robuste Merkmale des Korpus -, während sich der Rest neu mischt. Berichte nur Schlussfolgerungen, die eine solche Prüfung überstehen, und behandle die Anzahl der Topics wie *k* beim Clustering: als eine Detailebene, die du wählst und durch Lesen begründest.

## Zusammenfassung

| Methode | Liefert | Wichtigste Einschränkung |
|---|---|---|
| Kosinus über TF-IDF | Ähnlichkeit zwischen zwei beliebigen Dokumenten | blind für Synonyme und Paraphrasen |
| Schwellenwert durch Lesen | eine begründbare Grenze für "dieselbe Geschichte" | ist eine Ermessensentscheidung - dokumentiere sie |
| LSA | dichte, niedrigdimensionale Vektoren | Dimensionen nur teilweise interpretierbar |
| k-means | eine Gruppe pro Dokument | jedes Dokument wird in eine Gruppe gezwungen |
| NMF / LDA | Topic-Mischungen pro Dokument | instabil zwischen Durchläufen; LDA schwach bei kurzen Texten |

Alle diese Methoden sehen Text als Bag of Words. Skript 05 behält den Bag of Words bei, fügt aber Labels hinzu, und Skript 06 ersetzt ihn durch Repräsentationen, die wissen, welche Wörter Ähnliches bedeuten.

## Übungen

1. **Story Matching, überprüft.** Ziehe 30 zufällige zugeordnete Paare beim gewählten Schwellenwert und markiere jedes als richtig oder falsch. Wie hoch ist die Precision? Wie gut sagt die Übereinstimmung der Ressortgruppen deine Urteile voraus?
2. **Nur Schlagzeilen.** Schätze das NMF-Modell auf `title_lemmas` statt auf `text_lemmas` neu. Welche Topics überleben ohne Teaser?
3. **Die Anzahl der Topics wählen.** Schätze NMF mit 8, 15 und 30 Topics. Ab welcher Größe zerfallen Topics in Beinahe-Duplikate?
4. **Topics im Zeitverlauf.** Zeichne das monatliche mittlere Gewicht zweier NMF-Topics, getrennt nach Zeitung. Passen die Spitzen zu den Ausbrüchen (Bursts), die wir im Skript 03 gefunden haben?

## Quellen

* https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
* https://scikit-learn.org/stable/modules/clustering.html
* https://scikit-learn.org/stable/modules/decomposition.html
* https://de.wikipedia.org/wiki/Tf-idf-Ma%C3%9F
* https://spacy.io/models/de